# EvalHub SDK Demo

This notebook demonstrates the EvalHub Python SDK for programmatic LLM evaluation.

EvalHub orchestrates evaluation jobs using providers like lm-evaluation-harness, Garak, GuideLLM, and LightEval,
with results automatically tracked in MLflow.

**Prerequisites:**
- EvalHub deployed in your namespace (run `deploy.sh` first)
- A model deployed via InferenceService
- `disableLMEval: false` in OdhDashboardConfig

In [ ]:
# Install the EvalHub SDK
!pip install -q "eval-hub-sdk[client]"

In [ ]:
import os
import json
import time

# --- Configuration ---
# Set these to match your deployment
EVALHUB_URL = os.environ.get("EVALHUB_URL", "https://evalhub.redhat-ods-applications.svc:8443")
NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
TENANT = NAMESPACE

# Model endpoint — prefer direct (non-MaaS) external route for EvalHub adapter compatibility
MODEL_NAME = os.environ.get("DIRECT_MODEL_NAME", os.environ.get("MODEL_NAME", ""))
MODEL_URL = os.environ.get("DIRECT_ROUTE_URL", os.environ.get("MODEL_URL", ""))

print(f"EvalHub URL: {EVALHUB_URL}")
print(f"Namespace/Tenant: {TENANT}")

## 1. Connect to EvalHub

In [ ]:
# eval-hub-sdk >= 0.4: SyncEvalHubClient (EvalHubClient is the async client) with resource
# sub-APIs: client.providers / benchmarks / collections / jobs
from evalhub import (SyncEvalHubClient, ModelConfig, BenchmarkConfig, CollectionRef,
                     JobSubmissionRequest, ExperimentConfig, JobStatus)

# EvalHub server may omit 'description'/'category' on benchmarks; relax the SDK model (same as english-benchmark)
from evalhub.models.api import Benchmark as _Benchmark, Provider as _Provider, ProviderList as _ProviderList
for _f in ("description", "category"):
    if _f in _Benchmark.model_fields:
        _Benchmark.model_fields[_f].default = None
        _Benchmark.model_fields[_f].annotation = str | None
_Benchmark.model_rebuild(force=True); _Provider.model_rebuild(force=True); _ProviderList.model_rebuild(force=True)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=os.environ.get("EVALHUB_AUTH_TOKEN"),
    insecure=True,
    tenant=TENANT,
)

health = client.health()
print(f"Status:  {health.status}")
print(f"Version: {health.version}")


## 2. Discover Providers and Benchmarks

In [ ]:
# List available providers
providers = client.providers.list()
print("Available providers:")
for p in providers:
    pid = getattr(getattr(p, "resource", None), "id", None) or getattr(p, "id", "?")
    print(f"  - {pid}: {getattr(p, 'name', '')} ({len(getattr(p, 'benchmarks', []) or [])} benchmarks)")


In [ ]:
# List benchmarks for lm-evaluation-harness
benchmarks = client.benchmarks.list(provider_id="lm_evaluation_harness")
print(f"\nlm-evaluation-harness benchmarks ({len(benchmarks)}):")
for b in benchmarks[:15]:
    print(f"  [{getattr(b, 'category', None) or 'general'}] {b.id}: {getattr(b, 'description', '') or ''}")
if len(benchmarks) > 15:
    print(f"  ... and {len(benchmarks) - 15} more")


In [ ]:
# List available collections (pre-built benchmark suites)
collections = client.collections.list()
print("Available collections:")
for c in collections:
    cid = getattr(getattr(c, "resource", None), "id", None) or getattr(c, "id", "?")
    bms = getattr(c, "benchmarks", []) or []
    print(f"  - {cid}: {len(bms)} benchmarks")
    for b in bms[:5]:
        print(f"      {getattr(b, 'provider_id', '?')}/{getattr(b, 'id', '?')}")


## 3. Auto-Detect Model Endpoint

In [ ]:
import subprocess

# Env vars injected via notebook-env ConfigMap; fallback for standalone use
if not MODEL_NAME or not MODEL_URL:
    # Fallback: detect InferenceService external route URL
    try:
        result = subprocess.run(
            ["oc", "get", "inferenceservice", "-A", "--no-headers",
             "-o", "custom-columns=NS:.metadata.namespace,NAME:.metadata.name,URL:.status.url"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.strip().split("\n"):
                parts = line.split()
                if len(parts) >= 3 and "qwen" in parts[1].lower():
                    MODEL_NAME = MODEL_NAME or parts[1]
                    route_url = parts[2].rstrip("/")
                    MODEL_URL = MODEL_URL or f"{route_url}/v1"
                    break
    except Exception:
        pass

if not MODEL_NAME or not MODEL_URL:
    # No interactive prompt: notebooks also run headless (jupyter nbconvert --execute)
    raise RuntimeError("Model not found. Set MODEL_NAME and MODEL_URL (e.g. https://model-route.apps.cluster/v1) "
                       "in the notebook-env ConfigMap or the cell above, or deploy an InferenceService.")

print(f"Model: {MODEL_NAME}")
print(f"URL: {MODEL_URL}")

## 4. Submit an Evaluation Job

Run a single benchmark (MMLU) against the model.

In [ ]:
# Submit an MMLU evaluation. LIMIT (injected by deploy.sh, default 20) keeps the demo quick;
# unset it for a full run. The model is the in-cluster predictor URL, so no auth is needed.
LIMIT = int(os.environ.get("LIMIT", "20"))
model = ModelConfig(url=MODEL_URL, name=MODEL_NAME)
request = JobSubmissionRequest(
    name=f"evalhub-demo-mmlu-{MODEL_NAME}",
    model=model,
    benchmarks=[BenchmarkConfig(id="mmlu", provider_id="lm_evaluation_harness",
                                parameters={"num_fewshot": 0, "limit": LIMIT})],
    experiment=ExperimentConfig(name=f"evalhub-demo-{MODEL_NAME}"),
)
job = client.jobs.submit(request)
job_id = job.resource.id
print(f"Job submitted: {job_id}")
print(f"Status: {job.status.state.value if job.status else 'pending'}")


In [ ]:
# Poll for job completion
print(f"Waiting for job {job_id}...")
job = client.jobs.wait_for_completion(job_id, timeout=int(os.environ.get("EVAL_TIMEOUT", "1800")), poll_interval=15)
state = job.status.state.value if job.status else "unknown"
print(f"Final status: {state}")


## 5. Retrieve Results

In [ ]:
print(f"Job: {job_id}")
print(f"Model: {MODEL_NAME}")
print(f"Status: {state}")
print("\nResults:")
print("-" * 60)
results = job.results.model_dump(exclude_none=True) if job.results else {}
print(json.dumps(results, indent=2, default=str)[:3000] or "  (no results)")


## 6. Run a Collection (Benchmark Suite)

Collections group benchmarks from multiple providers into reusable evaluation suites.

In [ ]:
# Submit a safety evaluation using the safety-and-fairness collection (runs several benchmarks;
# this cell only submits -- follow progress in the dashboard: Develop and train > Evaluations)
safety_job = client.jobs.submit(JobSubmissionRequest(
    name=f"safety-eval-{MODEL_NAME}",
    model=model,
    collection=CollectionRef(id="safety-and-fairness-v1", benchmarks=[]),
    experiment=ExperimentConfig(name=f"safety-eval-{MODEL_NAME}"),
))
print(f"Safety evaluation job submitted: {safety_job.resource.id}")


## 7. List All Jobs

In [ ]:
jobs = client.jobs.list(limit=20)
print(f"Jobs: {len(jobs)}")
print(f"{'ID':<38} {'Name':<40} {'State'}")
print("-" * 90)
for j in jobs:
    st = j.status.state.value if j.status else "?"
    print(f"{j.resource.id:<38} {j.name[:39]:<40} {st}")


## Summary

This notebook demonstrated:

1. **Connecting** to EvalHub via the Python SDK
2. **Discovering** available providers, benchmarks, and collections
3. **Submitting** individual benchmark evaluations
4. **Running** pre-built collection suites (safety, leaderboard)
5. **Retrieving** and displaying results

### Next Steps

- View results in MLflow: experiment tracking with metrics and artifacts
- Use the dashboard: **Develop and train > Evaluations** for visual job management
- CLI: `evalhub submit --model ... --benchmark ...` for terminal workflows
- Custom adapters: extend EvalHub with your own evaluation frameworks